I am using OpenAlex to gather papers and citations

In [ ]:
import requests
import os
import signal
import time
from sqlalchemy import func, text
from sqlalchemy.dialects.postgresql import insert
from datetime import datetime
from database.database import session, init_db
from database.models.paper import Paper
from database.models.citation import Citation
from dotenv import load_dotenv

load_dotenv()
init_db()

In [ ]:
class OpenAlexApiRequest:
    """
    Handles API requests to the OpenAlex 'works' endpoint.
    Retrieves and parses academic papers based on concept IDs.
    """
    BASE_URL = "https://api.openalex.org/works"
    MAX_PER_PAGE = 200

    def __init__(self, concept_ids, per_page=200) -> None:
        """
        Initialize the API request handler.

        Args:
            concept_ids (list): List of OpenAlex concept IDs to filter papers.
            per_page (int, optional): Number of results per API page. Defaults to 200.
        """
        self.concept_ids = concept_ids  # e.g., ["C154945302", "C154954719"]
        self.per_page = min(per_page, self.MAX_PER_PAGE)
        self.api_key = os.getenv("OPEN_ALEX_API_KEY", "")
        self.email = os.getenv("OPEN_ALEX_EMAIL", "")

    def _build_filters(self, from_date=None, min_references=15, min_cited_by=15):
        """
        Build the filter strings for the OpenAlex API query.

        Args:
            from_date (str, optional): The earliest publication date to fetch.
            min_references (int, optional): Minimum number of references required. Defaults to 15.
            min_cited_by (int, optional): Minimum number of citations required. Defaults to 15.

        Returns:
            str: The joined filter string to use in the API parameters.
        """
        ids_string = "|".join(self.concept_ids)

        filters = (
            f"concepts.id:{ids_string}"
            f",referenced_works_count:>{min_references}"
            f",cited_by_count:>{min_cited_by}"
            f",language:en"
            f",has_abstract:true"
        )
        if from_date:
            filters += f",from_publication_date:{from_date}"
        return filters

    def _build_params(self, cursor="*", from_date=None):
        """
        Build the full parameters dictionary for the API request.

        Args:
            cursor (str, optional): Pagination cursor. Defaults to "*".
            from_date (str, optional): The earliest publication date to fetch.

        Returns:
            dict: The parameters dictionary.
        """
        params = {
            "filter": self._build_filters(from_date=from_date),
            "per_page": self.per_page,
            "sort": "publication_date:asc",
            "select": ",".join([
                "id", "doi", "title", "abstract_inverted_index",
                "referenced_works", "referenced_works_count",
                "publication_date", "cited_by_count", "type", "concepts",
            ]),
            "cursor": cursor,
        }
        if self.email:
            params["mailto"] = self.email
        return params

    def fetch_page(self, cursor="*", from_date=None):
        """
        Fetch a single page of results from the OpenAlex API.

        Args:
            cursor (str, optional): Pagination cursor. Defaults to "*".
            from_date (str, optional): The earliest publication date to fetch.

        Returns:
            tuple: A tuple containing:
                - results (list): List of raw paper data dictionaries.
                - next_cursor (str): Cursor for the next page, or None.
                - total_count (int): Total number of available results.
        """
        params = self._build_params(cursor=cursor, from_date=from_date)
        headers = {"Authorization": f"Bearer {self.api_key}"} if self.api_key else {}

        response = requests.get(self.BASE_URL, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = data.get("results", [])
        meta = data.get("meta", {})
        next_cursor = meta.get("next_cursor")
        total_count = meta.get("count", 0)

        return results, next_cursor, total_count

    @staticmethod
    def reconstruct_abstract(inverted_index):
        """
        Reconstruct the full abstract text from an inverted index structure.

        Args:
            inverted_index (dict): Dictionary mapping words to lists of their positions.

        Returns:
            str: The reconstructed abstract, or None if the input is empty/None.
        """
        if not inverted_index:
            return None
        
        word_positions = []
        
        for word, positions in inverted_index.items():
            for pos in positions:
                word_positions.append((pos, word))
        
        word_positions.sort(key=lambda x: x[0])
        return " ".join(word for _, word in word_positions)

    @staticmethod
    def extract_id(openalex_url):
        """
        Extract the plain ID string from an OpenAlex URL.

        Args:
            openalex_url (str): The full OpenAlex entity URL.

        Returns:
            str: The extracted ID, or None if the URL is empty.
        """
        if not openalex_url:
            return None
        return openalex_url.split("/")[-1]

    @staticmethod
    def parse_date(date_str):
        """
        Parse a date string into a datetime.date object.

        Args:
            date_str (str): Date string in 'YYYY-MM-DD' format.

        Returns:
            datetime.date: The parsed date object, or None if parsing fails.
        """
        if not date_str:
            return None
        try:
            return datetime.strptime(date_str, "%Y-%m-%d").date()
        except ValueError:
            return None

    @staticmethod
    def parse_concepts(concepts_list):
        """
        Extract concept ID, name, level, and score from raw OpenAlex concepts list.

        Args:
            concepts_list (list): List of concept dictionaries from OpenAlex.

        Returns:
            list: Parsed list of concept dictionaries, or None if the input is empty.
        """
        if not concepts_list:
            return None
        return [
            {
                "id": c.get("id", "").split("/")[-1],
                "name": c.get("display_name"),
                "level": c.get("level"),
                "score": c.get("score"),
            }
            for c in concepts_list
        ]

In [ ]:
class PaperCollector:
    """
    Coordinates the process of fetching papers from OpenAlex and saving them 
    into the database, including handling pagination, batching, and graceful stopping.
    """
    RATE_LIMIT_DELAY = 0.2
    DEFAULT_START_DATE = "1950-01-01"
    BATCH_SIZE = 2000

    def __init__(self, concept_ids) -> None:
        """
        Initialize the PaperCollector.

        Args:
            concept_ids (list): List of OpenAlex concept IDs to filter papers.
        """
        self.session = session
        self.api = OpenAlexApiRequest(concept_ids=concept_ids)
        self.concept_ids = set(concept_ids)
        self._stop_requested = False

        # Tuning: Disable synchronous commit for faster bulk loading
        self.session.execute(text("SET synchronous_commit TO OFF"))

        signal.signal(signal.SIGINT, self._handle_signal)

    def _handle_signal(self, signum, frame):
        """
        Handle interrupt signals (e.g., Ctrl+C) to trigger a graceful shutdown.
        """
        print("\nGraceful shutdown requested. Finishing current batch...")
        self._stop_requested = True

    def get_resume_date(self):
        """
        Retrieve the most recent paper publication date from the database to resume collection.

        Returns:
            str: The most recent date in ISO format, or the default start date if none exist.
        """
        last_date = self.session.query(func.max(Paper.publication_date)).scalar()
        if last_date:
            return last_date.isoformat()
        return self.DEFAULT_START_DATE

    def _parse_paper(self, raw):
        """
        Parse raw paper data from OpenAlex into database-ready formats.

        Args:
            raw (dict): Raw dictionary representing a paper from OpenAlex.

        Returns:
            tuple: A tuple containing:
                - paper_dict (dict or None): Dictionary of parsed paper attributes, or None if invalid.
                - citation_dicts (list): List of parsed citation mappings.
        """
        paper_id = self.api.extract_id(raw.get("id"))
        if not paper_id:
            return None, []

        abstract = self.api.reconstruct_abstract(raw.get("abstract_inverted_index"))
        if not abstract:
            return None, []
            
        # Filter out very short abstracts
        if len(abstract) < 70:
            return None, []

        paper_dict = {
            "paperId": paper_id,
            "doi": raw.get("doi"),
            "title": raw.get("title"),
            "abstract": abstract,
            "publication_date": self.api.parse_date(raw.get("publication_date")),
            "cited_by_count": raw.get("cited_by_count"),
            "paper_type": raw.get("type"),
            "concepts": raw.get("concepts"), 
        }

        citation_dicts = []
        for ref_url in raw.get("referenced_works", []):
            ref_id = self.api.extract_id(ref_url)
            if ref_id:
                citation_dicts.append({"source_paper_id": paper_id, "target_paper_id": ref_id})

        return paper_dict, citation_dicts

    def _save_batch(self, papers_data):
        """
        Parse and save a batch of papers and their citations to the database.

        Args:
            papers_data (list): List of raw paper data dictionaries.

        Returns:
            int: The number of valid papers successfully processed and saved.
        """
        all_papers = []
        all_citations = []

        for raw in papers_data:
            paper_dict, citation_dicts = self._parse_paper(raw)
            if not paper_dict:
                continue
            all_papers.append(paper_dict)
            all_citations.extend(citation_dicts)

        if all_papers:
            stmt = insert(Paper).values(all_papers)
            stmt = stmt.on_conflict_do_update(
                index_elements=["paperId"],
                set_={
                    "doi": stmt.excluded.doi,
                    "title": stmt.excluded.title,
                    "abstract": stmt.excluded.abstract,
                    "publication_date": stmt.excluded.publication_date,
                    "cited_by_count": stmt.excluded.cited_by_count,
                    "paper_type": stmt.excluded.paper_type,
                    "concepts": stmt.excluded.concepts,
                },
            )
            self.session.execute(stmt)

        if all_citations:
            stmt = insert(Citation).values(all_citations)
            stmt = stmt.on_conflict_do_nothing(
                index_elements=["source_paper_id", "target_paper_id"],
            )
            self.session.execute(stmt)

        self.session.commit()
        return len(all_papers)

    def collect(self, max_papers=None):
        """
        Start or resume the collection of papers from OpenAlex.
        Fetches papers, buffers them into batches, and saves them to the database.

        Args:
            max_papers (int, optional): An optional limit on the maximum number of papers to gather.
        """
        self._stop_requested = False
        resume_date = self.get_resume_date()
        print(f"Starting collection from {resume_date}")

        cursor = "*"
        total_saved = 0
        page = 0
        
        results_buffer = []

        while cursor:
            if self._stop_requested:
                print("Stop requested. Saving buffer...")
                if results_buffer:
                    saved = self._save_batch(results_buffer)
                    total_saved += saved
                    print(f"Buffer saved: {saved} papers.")
                break

            try:
                results, cursor, total_available = self.api.fetch_page(cursor=cursor, from_date=resume_date)
            except Exception as e:
                print(f"Error: {e}. Committing progress and stopping.")
                if results_buffer:
                    self._save_batch(results_buffer)
                self.session.commit()
                break

            if not results:
                break

            # Buffer results instead of saving immediately
            results_buffer.extend(results)
            page += 1
            
            # Flush buffer when it gets large enough
            if len(results_buffer) >= self.BATCH_SIZE:
                saved = self._save_batch(results_buffer)
                total_saved += saved
                
                # Stats for the latest batch
                first_date = results_buffer[0].get("publication_date", "?")
                last_date = results_buffer[-1].get("publication_date", "?")
                print(f"Pages processed: {page} | Batch saved {saved} papers | dates {first_date} -> {last_date} | total: {total_saved}")
                
                results_buffer = []

            if max_papers and total_saved >= max_papers:
                print(f"Reached limit of {max_papers}")
                break

            time.sleep(self.RATE_LIMIT_DELAY)

        # Flush any remaining items in buffer at the end
        if results_buffer and not self._stop_requested:
             saved = self._save_batch(results_buffer)
             total_saved += saved
             print(f"Final buffer saved: {saved} papers.")

        print(f"Done. Total papers saved this run: {total_saved}")

In [ ]:
ml_concepts = [
    "C119857082",  # Machine Learning (You had the AI ID here)
    "C154945302",  # Artificial Intelligence (This is the parent concept)
    "C204321447",  # Natural Language Processing (NLP)
    "C108583219",  # Deep Learning (Verified ID)
]

collector = PaperCollector(concept_ids=ml_concepts)
collector.collect()